In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abdelkrimboucerba22/datatra/.pdf
/kaggle/input/datasets/abdelkrimboucerba22/datatra2/.pdf


In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/abdelkrimboucerba22/datatra/.pdf
/kaggle/input/datasets/abdelkrimboucerba22/datatra2/.pdf


In [3]:
!pip install pymupdf openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 69.3 MB/s eta 0:00:00:00:0100:01


In [9]:
import fitz
import base64
from openai import OpenAI
import os
import glob


client = OpenAI(
    api_key="YOUR_API_KEY_HERE",
    base_url="http://app.ai-grid.io:4000/v1"
)

def extract_text_from_pdf(pdf_path, output_txt_path):
    doc = fitz.open(pdf_path)
    extracted_text = ""
    total_pages = len(doc)
    
    print(f"File: {os.path.basename(pdf_path)}")
    print(f"Pages: {total_pages}")
    print("Processing...\n")

    for page_num in range(total_pages):
        print(f"   Page {page_num + 1}/{total_pages}...", end=" ")
        
        page = doc.load_page(page_num)
        
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        img_bytes = pix.tobytes("png")
        base64_image = base64.b64encode(img_bytes).decode('utf-8')

        try:
            response = client.chat.completions.create(
                model="deepseek-ocr",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text", 
                                "text": "Extract Arabic text from this image with high accuracy and preserve paragraph formatting. Do not add any comments, only the extracted text."
                            },
                            {
                                "type": "image_url",
                                "image_url": {"url": f"data:image/png;base64,{base64_image}"}
                            }
                        ]
                    }
                ]
            )
            
            page_text = response.choices[0].message.content
            extracted_text += f"\n\n--- Page {page_num + 1} ---\n\n" + page_text
            print("Done")
            
        except Exception as e:
            print(f"Error: {e}")

    with open(output_txt_path, "w", encoding="utf-8") as f:
        f.write(extracted_text)
        
    print(f"\nSaved: {output_txt_path}\n")
    print("=" * 50)

print("Searching for PDF files...\n")
pdf_files = glob.glob("/kaggle/input/**/*.pdf", recursive=True)

if not pdf_files:
    print("No PDF files found!")
    print("Make sure to add Dataset to the Notebook")
else:
    print(f"Found {len(pdf_files)} file(s)\n")
    
    for pdf_path in pdf_files:
        base_name = os.path.splitext(os.path.basename(pdf_path))[0]
        output_path = f"/kaggle/working/{base_name}.txt"
        
        extract_text_from_pdf(pdf_path, output_path)
    
    print("\nDone!")
    print("\nOutput files in: /kaggle/working/")

Searching for PDF files...

No PDF files found!
Make sure to add Dataset to the Notebook


In [8]:
!pip install pymupdf pytesseract pillow -q
!apt-get update -qq && apt-get install -y tesseract-ocr tesseract-ocr-ara -qq

import fitz
import pytesseract
from PIL import Image
import os

def extract_text_from_scanned_pdf(pdf_path, output_txt_path):
    doc = fitz.open(pdf_path)
    extracted_text = ""
    total_pages = len(doc)
    
    print(f"File: {os.path.basename(pdf_path)}")
    print(f"Pages: {total_pages}")
    print("Processing...\n")

    for page_num in range(total_pages):
        print(f"   Page {page_num + 1}/{total_pages}...", end=" ")
        
        page = doc.load_page(page_num)
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        
        try:
            page_text = pytesseract.image_to_string(img, lang='ara')
            extracted_text += f"\n\n--- Page {page_num + 1} ---\n\n" + page_text
            print("Done")
        except Exception as e:
            print(f"Error: {e}")

    with open(output_txt_path, "w", encoding="utf-8") as f:
        f.write(extracted_text)
        
    print(f"\nSaved: {output_txt_path}\n")
    print("=" * 50)

pdf_files = [
    ("/kaggle/input/datasets/abdelkrimboucerba22/datatra/.pdf", "/kaggle/working/result_1.txt"),
    ("/kaggle/input/datasets/abdelkrimboucerba22/datatra2/.pdf", "/kaggle/working/result_2.txt")
]

print("Starting...\n")

for pdf_path, output_path in pdf_files:
    if os.path.exists(pdf_path):
        extract_text_from_scanned_pdf(pdf_path, output_path)
    else:
        print(f"Not found: {pdf_path}")

print("\nDone!")
print("Output: /kaggle/working/")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Starting...

File: .pdf
Pages: 404
Processing...

   Page 1/404... Done
   Page 2/404... Done
   Page 3/404... Done
   Page 4/404... Done
   Page 5/404... 

KeyboardInterrupt: 